In [1]:
import os

# 1. Deteksi environment (Colab vs Lokal/VS Code)
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    base_dir = "/content"
else:
    base_dir = os.getcwd()

# 2. Clone atau Sinkronisasi Repo Github
repo_url = "https://github.com/gacoanphilia/Document-Image-Classification.git"
repo_dir = os.path.join(base_dir, "Document-Image-Classification")

if not os.path.exists(repo_dir):
    print("Mengunduh repo baru (Clone)...")
    !git clone {repo_url}
else:
    print("Repo ditemukan, menyinkronkan pembaruan (Pull)...")
    %cd {repo_dir}
    !git pull origin main

%cd {repo_dir}
!mkdir -p data

# 3. Ekstrak data jika berada di Colab
if IN_COLAB:
    zip_path = "/content/drive/MyDrive/Semester 5/SD4504_SML/Document-Image-Classification/data/document-image-classification-sd234504.zip"
    print("Mengekstrak ZIP ke folder 'data/'...")
    !unzip -q -o "{zip_path}" -d data/
else:
    print("Eksekusi Lokal: Pastikan file test.csv dan folder test/ berada di dalam folder 'data/'.")

print("Setup Environment Selesai!")

Mounted at /content/drive
Mengunduh repo baru (Clone)...
Cloning into 'Document-Image-Classification'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 83 (delta 16), reused 41 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 1.47 MiB | 12.61 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/Document-Image-Classification
Mengekstrak ZIP ke folder 'data/'...
Setup Environment Selesai!


In [9]:
import sys
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import efficientnet_b0
from PIL import Image
from tqdm import tqdm

# Hardware Routing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device untuk Inferensi: {device}")

# Path Setup
DATA_DIR = Path("data").resolve()
TEST_CSV = DATA_DIR / "test.csv"
TEST_IMG_DIR = DATA_DIR / "test"
SAMPLE_SUB = DATA_DIR / "sample_submission.csv"

# Path model dari 02_modeling (Pencarian fleksibel di folder Colab / Lokal)
MODEL_PATH = Path("data").resolve() / "models" / "best_model.pth"
if not MODEL_PATH.exists():
    MODEL_PATH = Path("models") / "best_model.pth"
if not MODEL_PATH.exists():
    MODEL_PATH = Path("/content/models/best_model.pth")

Compute Device untuk Inferensi: cuda


In [7]:
# Transformasi khusus inferensi (Konsisten dengan val_transform pada 02_modeling)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class DocumentTestDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = Path(img_dir)
        self.transform = transform

        # Deteksi kolom nama file gambar secara otomatis
        if 'filename' in self.df.columns:
            self.file_col = 'filename'
        elif 'id' in self.df.columns:
            self.file_col = 'id'
        else:
            self.file_col = self.df.columns[0]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = str(self.df.iloc[idx][self.file_col])
        img_path = self.img_dir / img_name

        # Penanganan opsional jika ekstensi file tidak tertulis di CSV
        if not img_path.exists():
            for ext in ['.jpg', '.png', '.tif', '.jpeg']:
                if (self.img_dir / f"{img_name}{ext}").exists():
                    img_path = self.img_dir / f"{img_name}{ext}"
                    break

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return image, img_name

test_dataset = DocumentTestDataset(csv_file=TEST_CSV, img_dir=TEST_IMG_DIR, transform=test_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print(f"Total gambar data test dimuat: {len(test_dataset)} sampel ({len(test_loader)} batch)")

Total gambar data test dimuat: 7994 sampel (250 batch)


In [10]:
NUM_CLASSES = 16

# Inisialisasi Arsitektur EfficientNet-B0
model = efficientnet_b0(weights=None)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

# Memuat Trained Checkpoint
if MODEL_PATH.exists():
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    print(f"Berhasil memuat bobot dari: {MODEL_PATH}")
else:
    raise FileNotFoundError(f"File '{MODEL_PATH}' tidak ditemukan! Selesaikan training di 02_modeling.ipynb terlebih dahulu.")

model = model.to(device)
model.eval() # Matikan Dropout dan Batch Normalization update

Berhasil memuat bobot dari: models/best_model.pth


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [11]:
all_predictions = []
all_img_names = []

print("Memulai proses inferensi pada data test...")
with torch.no_grad():
    for images, img_names in tqdm(test_loader, desc="Inference"):
        images = images.to(device)
        outputs = model(images)

        # Ambil indeks kelas dengan logits tertinggi
        _, preds = torch.max(outputs, 1)

        all_predictions.extend(preds.cpu().numpy())
        all_img_names.extend(img_names)

print("Proses inferensi selesai!")

Memulai proses inferensi pada data test...


Inference: 100%|██████████| 250/250 [01:27<00:00,  2.86it/s]

Proses inferensi selesai!


In [13]:
# Membaca format bawaan sample_submission.csv
sub_df = pd.read_csv(SAMPLE_SUB)

id_col = sub_df.columns[0]
label_col = sub_df.columns[1] if len(sub_df.columns) > 1 else 'target'

# Pemetaan hasil prediksi berdasarkan nama gambar
pred_map = dict(zip(all_img_names, all_predictions))

# Map ID ke prediksi, gunakan Series untuk fallback jika ada nilai NaN
mapped_series = sub_df[id_col].astype(str).map(pred_map)
sub_df[label_col] = mapped_series.fillna(pd.Series(all_predictions, index=sub_df.index))

# Simpan ke CSV
OUTPUT_SUB_PATH = Path("submission.csv")
sub_df.to_csv(OUTPUT_SUB_PATH, index=False)

print(f"File submission berhasil diekspor ke: {OUTPUT_SUB_PATH.resolve()}")
print("\nPreview 10 baris pertama submission:")
print(sub_df.head(10))

print("\nDistribusi Prediksi Kelas:")
print(sub_df[label_col].value_counts().sort_index())

File submission berhasil diekspor ke: /content/Document-Image-Classification/submission.csv

Preview 10 baris pertama submission:
           id  label
0  test_00001    0.0
1  test_00002    0.0
2  test_00003    6.0
3  test_00004   11.0
4  test_00005   13.0
5  test_00006    1.0
6  test_00007    9.0
7  test_00008   12.0
8  test_00009    8.0
9  test_00010   15.0

Distribusi Prediksi Kelas:
label
0.0     574
1.0     483
2.0     520
3.0     563
4.0     533
5.0     520
6.0     522
7.0     512
8.0     464
9.0     437
10.0    456
11.0    408
12.0    491
13.0    475
14.0    560
15.0    476
Name: count, dtype: int64
